In [ ]:
# need to do first:
# normalize columns: total demand, mode score, region location (1-10) V
# don't forget, total demand - use log, then filter 0 and then normalize 1-10
# weight pop and emp columns, then normalize (1-10)
# weight bus terminal column, then normalize (1-10)

# all normalizations should be within the hub type group

In [ ]:
import pandas as pd
import numpy as np

import os

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Hubs/grouped_hubs_ready_for_scoring_29102025.csv', encoding='utf-8')

In [ ]:
df.head(1)

In [ ]:
df.drop(columns='Unnamed: 0', inplace=True)
df.set_index('group', inplace=True)

In [ ]:
df.head(1)

In [ ]:
for i, row in df.iterrows():
  if '400380' in row['node']:
    print(i)

# Filter out groups with less than 1000 total demand

In [ ]:
df = df[df['TotalDemand']>=1000]

In [ ]:
def make_list(txt):
  txt = txt.replace('[','').replace(']','').replace("'","").split(',')
  return txt

In [ ]:
df['Mode_Planned'] = df['Mode_Planned'].apply(lambda x: make_list(x))

In [ ]:
df = df[df['Mode_Planned'].apply(lambda x: len(x) > 1)]

In [ ]:
df

In [ ]:
df.columns

### Normalize Region Location and Mode score

In [ ]:
def norm_col(df, col) -> pd.DataFrame:
  min_val = df[col].min()
  max_val = df[col].max()

  df[col + '_Norm'] = 1 + (df[col] - min_val) * (10 - 1) / (max_val - min_val)

  return df

In [ ]:
cols_to_normalize = ['RegionLocation', 'score','bus_terminal']

In [ ]:
for hub in df['HubType'].unique():
  for col in cols_to_normalize:
    df.loc[df['HubType'] == hub, col + '_Norm'] = norm_col(df[df['HubType'] == hub].copy(), col)[col + '_Norm']

In [ ]:
df['score_Norm'].max()

In [ ]:
df[['score','score_Norm']].loc[777,:]

## Normalize total deman

In [ ]:
def norm_demand(df, col) -> pd.DataFrame:
  # calculate LogTotalDemand_Daily and assign to a new column
  df['Log' + col] = df[col].apply(lambda x: 0 if x == 0 else np.log(x))

  # filter out rows where 'Log' + col is greater than 0
  filtered_df = df[df['Log' + col] > 0]

  # if filtered_df is empty assign min and max to 0 to avoid errors and return df with added columns.
  if filtered_df.empty:
    df['Log' + col + '_Norm'] = 0
    return df

  # calculate min and max from the 'Log' + col column of the filtered dataframe
  min_val = filtered_df['Log' + col].min()
  max_val = filtered_df['Log' + col].max()

  # calculate and assign LogTotalDemand_Daily_Norm values
  df['Log' + col + '_Norm'] = 1 + (df['Log' + col] - min_val) * (10 - 1) / (max_val - min_val)

  return df

In [ ]:
col = 'TotalDemand'
for hub in df['HubType'].unique():
  # df.loc[df['HubType'] == hub, col + '_Norm'] = norm_col(df[df['HubType'] == hub].copy(), col)[col + '_Norm']
  df.loc[df['HubType'] == hub, col + '_Norm'] = norm_demand(df[df['HubType'] == hub].copy(), col)['Log' + col + '_Norm']

In [ ]:
df.columns

In [ ]:
for hub in df['HubType'].unique():
  print(df[df['HubType']==hub]['TotalDemand_Norm'].min())

In [ ]:
df.columns

## Weight and normalize Pop and Emp columns

In [ ]:
df.columns

In [ ]:
pop_emp_cols = ['pop_0_500','emp_0_500', 'pop_500_1000', 'emp_500_1000', 'pop_1000_1500','emp_1000_1500']
pop_emp_weights = {k: 0 for k in pop_emp_cols}

In [ ]:
def hub_scores_by_type(
    df,
    hubtype_col="HubType",
    pop_cols=("pop_0_500","pop_500_1000","pop_1000_1500"),
    emp_cols=("emp_0_500","emp_500_1000","emp_1000_1500"),
    radii=(0, 500, 1000, 1500),
    w_pop=0.5,
    w_emp=0.5,
    beta=1.5,
    normalize=True
):
    mids = np.array([(radii[i] + radii[i+1]) / 2 for i in range(len(radii)-1)], dtype=float)
    decay = mids ** beta

    results = []
    for hubtype, group in df.groupby(hubtype_col):

        pop = group.loc[:, list(pop_cols)].to_numpy(dtype=float)
        emp = group.loc[:, list(emp_cols)].to_numpy(dtype=float)

        if hubtype in ['National','Regional']:
          w_pop = 0.2
          w_emp = 0.8
        else:
          w_pop = 0.8
          w_emp = 0.2

        combined = w_pop * pop + w_emp * emp
        score = (combined / decay).sum(axis=1)

        out = group.copy()
        out["score"] = score
        if normalize:
            smin, smax = score.min(), score.max()
            out["PopEmp_Score_Norm"] = 1+ (score - smin) * (10-1) / (smax - smin) if smax > smin else 0.0
        results.append(out)

    return pd.concat(results, axis=0).sort_index()

In [ ]:
# pop_emp_weights['pop_0_500'] = 1
# pop_emp_weights['emp_0_500'] = 1
# pop_emp_weights['pop_500_1000'] = 2/3
# pop_emp_weights['emp_500_1000'] = 2/3
# pop_emp_weights['pop_1000_1500'] = 1/3
# pop_emp_weights['emp_1000_1500'] = 1/3

In [ ]:
scored = hub_scores_by_type(df, w_pop=0.5, w_emp=0.5, beta=1.5)

## Weight and normalize Bus terminal column

In [ ]:
scored

In [ ]:
def monte_carlo_scoring(df, num_simulations=10000):
    """
    Performs Monte Carlo simulation to score each HubType based on random weights.

    Args:
        df: DataFrame containing the hub data with normalized columns.
        num_simulations: Number of Monte Carlo simulations to run.

    Returns:
        A DataFrame with the original data and a new column for the average simulated score.
    """
    scoring_cols = ['RegionLocation_Norm', 'bus_terminal', 'score_Norm', 'TotalDemand_Norm', 'PopEmp_Score_Norm']
    results = []

    for hub_type in df['HubType'].unique():
        hub_df = df[df['HubType'] == hub_type].copy()
        # Initialize a column to store the sum of scores for each row
        hub_df['Sum_Simulated_Scores'] = 0.0

        for i in range(num_simulations):
            # Generate random weights that sum to 1 and no weight is greater than 0.5
            weights = np.random.rand(len(scoring_cols))
            while any(weights > 0.5):
              weights = np.random.rand(len(scoring_cols))
            weights /= weights.sum()


            # Calculate the weighted score for each row and add it to the sum
            hub_df['Sum_Simulated_Scores'] += (hub_df[scoring_cols] * weights).sum(axis=1)

        # Calculate the average score by dividing the sum by the number of simulations
        hub_df['Average_Simulated_Score'] = hub_df['Sum_Simulated_Scores'] / num_simulations

        # Drop the intermediate sum column
        hub_df.drop(columns='Sum_Simulated_Scores', inplace=True)

        results.append(hub_df)

    return pd.concat(results)

# Apply the Monte Carlo scoring function to the DataFrame
scored_mc = monte_carlo_scoring(scored.copy())

# Display the first few rows of the result
display(scored_mc.head())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def monte_carlo_scoring(df, num_simulations=10000):
    """
    Performs Monte Carlo simulation to score each HubType based on random weights.

    Args:
        df: DataFrame containing the hub data with normalized columns.
        num_simulations: Number of Monte Carlo simulations to run.

    Returns:
        A tuple containing:
        - DataFrame with the original data and a new column for the average simulated score
        - Dictionary with average weights for each scoring column
        - Dictionary with median weights for each scoring column
        - Dictionary with min weights for each scoring column
        - Dictionary with max weights for each scoring column
    """
    scoring_cols = ['RegionLocation_Norm', 'bus_terminal', 'score_Norm', 'TotalDemand_Norm', 'PopEmp_Score_Norm']
    results = []

    # Track weights across all simulations for statistics
    all_weights = []

    for hub_type in df['HubType'].unique():
        hub_df = df[df['HubType'] == hub_type].copy()
        # Initialize a column to store the sum of scores for each row
        hub_df['Sum_Simulated_Scores'] = 0.0

        for i in range(num_simulations):
            # Generate random weights that sum to 1 and no weight is greater than 0.5
            weights = np.random.rand(len(scoring_cols))
            weights /= weights.sum()  # Normalize first
            while any(weights > 0.5):  # Then check constraint after normalization
                weights = np.random.rand(len(scoring_cols))
                weights /= weights.sum()

            # Store weights for statistics calculation
            all_weights.append(weights.copy())

            # Calculate the weighted score for each row and add it to the sum
            hub_df['Sum_Simulated_Scores'] += (hub_df[scoring_cols] * weights).sum(axis=1)

        # Calculate the average score by dividing the sum by the number of simulations
        hub_df['Average_Simulated_Score'] = hub_df['Sum_Simulated_Scores'] / num_simulations

        # Drop the intermediate sum column
        hub_df.drop(columns='Sum_Simulated_Scores', inplace=True)

        results.append(hub_df)

    # Convert to numpy array for easier calculations
    all_weights = np.array(all_weights)

    # Calculate statistics for weights
    average_weights = np.mean(all_weights, axis=0)
    median_weights = np.median(all_weights, axis=0)
    min_weights = np.min(all_weights, axis=0)
    max_weights = np.max(all_weights, axis=0)

    avg_weight_dict = dict(zip(scoring_cols, average_weights))
    median_weight_dict = dict(zip(scoring_cols, median_weights))
    min_weight_dict = dict(zip(scoring_cols, min_weights))
    max_weight_dict = dict(zip(scoring_cols, max_weights))

    return pd.concat(results), avg_weight_dict, median_weight_dict, min_weight_dict, max_weight_dict

def export_weight_statistics_plot(avg_dict, min_dict, max_dict, filename='weight_statistics.png', figsize=(14, 8)):
    """
    Creates and exports a line plot showing average, min, and max weights for each scoring column.

    Args:
        avg_dict: Dictionary containing average weights for each column
        min_dict: Dictionary containing minimum weights for each column
        max_dict: Dictionary containing maximum weights for each column
        filename: Name of the PNG file to export
        figsize: Tuple for figure size (width, height)
    """
    # Convert to pandas Series (maintain original order)
    scoring_cols = list(avg_dict.keys())
    avg_series = pd.Series([avg_dict[col] for col in scoring_cols], index=scoring_cols)
    min_series = pd.Series([min_dict[col] for col in scoring_cols], index=scoring_cols)
    max_series = pd.Series([max_dict[col] for col in scoring_cols], index=scoring_cols)

    # Create the line plot
    plt.figure(figsize=figsize)
    x_pos = range(len(scoring_cols))

    # Plot lines for average, min, and max
    plt.plot(x_pos, avg_series.values, 'o-', color='blue', linewidth=3,
             markersize=8, label='Average Weight', alpha=0.8)
    plt.plot(x_pos, min_series.values, 's-', color='red', linewidth=2.5,
             markersize=7, label='Minimum Weight', alpha=0.7)
    plt.plot(x_pos, max_series.values, '^-', color='green', linewidth=2.5,
             markersize=7, label='Maximum Weight', alpha=0.7)

    # Customize the plot
    plt.xlabel('Scoring Columns', fontsize=12, fontweight='bold')
    plt.ylabel('Weight Value', fontsize=12, fontweight='bold')
    plt.title('Weight Statistics by Scoring Column (Average, Min, Max)', fontsize=14, fontweight='bold', pad=20)

    # Set x-axis labels with better formatting
    column_labels = [col.replace('_', ' ').replace('Norm', '(Norm)') for col in scoring_cols]
    plt.xticks(x_pos, column_labels, rotation=45, ha='right')

    # Add value labels on points
    for i, (avg_val, min_val, max_val) in enumerate(zip(avg_series.values, min_series.values, max_series.values)):
        plt.text(i, avg_val + 0.01, f'{avg_val:.3f}', ha='center', va='bottom',
                fontweight='bold', fontsize=9, color='blue')
        plt.text(i, min_val - 0.01, f'{min_val:.3f}', ha='center', va='top',
                fontweight='bold', fontsize=9, color='red')
        plt.text(i, max_val + 0.01, f'{max_val:.3f}', ha='center', va='bottom',
                fontweight='bold', fontsize=9, color='green')

    # Add horizontal line at expected average (0.2 for 5 columns)
    expected_avg = 1.0 / len(scoring_cols)
    plt.axhline(y=expected_avg, color='black', linestyle='--', alpha=0.5,
                label=f'Expected Average ({expected_avg:.3f})')

    # Add legend
    # plt.legend(loc='upper right', frameon=True, fancybox=True, shadow=True)

    # Add grid for better readability
    plt.grid(True, alpha=0.3, linestyle='--')

    # Set y-axis limits to show the full range nicely
    all_values = list(avg_series.values) + list(min_series.values) + list(max_series.values)
    y_margin = (max(all_values) - min(all_values)) * 0.1
    plt.ylim(min(all_values) - y_margin, max(all_values) + y_margin)

    # Adjust layout to prevent label cutoff
    plt.tight_layout()

    # Export as PNG
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Plot exported as {filename}")

    # Show the plot
    plt.show()

    # Return summary dataframe
    summary_df = pd.DataFrame({
        'Column': scoring_cols,
        'Average': avg_series.values,
        'Minimum': min_series.values,
        'Maximum': max_series.values,
        'Range': max_series.values - min_series.values
    })

    return summary_df
def export_score_distribution_plot(scored_df, filename='score_distribution.png', figsize=(14, 8)):
    """
    Creates and exports a line plot showing score distribution for each HubType.

    Args:
        scored_df: DataFrame with Monte Carlo scoring results
        filename: Name of the PNG file to export
        figsize: Tuple for figure size (width, height)
    """
    plt.figure(figsize=figsize)

    # Define colors for different HubTypes
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
              '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

    # Get unique HubTypes
    hub_types = sorted(scored_df['HubType'].unique())

    for i, hub_type in enumerate(hub_types):
        hub_data = scored_df[scored_df['HubType'] == hub_type]['Average_Simulated_Score']

        # Calculate histogram (density) for line plot
        counts, bin_edges = np.histogram(hub_data, bins=30, density=True)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

        # Plot line for this HubType
        plt.plot(bin_centers, counts,
                color=colors[i % len(colors)],
                linewidth=2.5,
                label=f'{hub_type} (n={len(hub_data)})',
                alpha=0.8)

    # Customize the plot
    plt.xlabel('Average Simulated Score', fontsize=12, fontweight='bold')
    plt.ylabel('Density', fontsize=12, fontweight='bold')
    plt.title('Score Distribution by Hub Type', fontsize=14, fontweight='bold', pad=20)

    # Add legend
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True, fancybox=True, shadow=True)

    # Add grid for better readability
    plt.grid(True, alpha=0.3, linestyle='--')

    # Adjust layout to prevent legend cutoff
    plt.tight_layout()

    # Export as PNG
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Plot exported as {filename}")

    # Show the plot
    plt.show()

    # Return summary statistics for each HubType
    summary_stats = {}
    for hub_type in hub_types:
        hub_data = scored_df[scored_df['HubType'] == hub_type]['Average_Simulated_Score']
        summary_stats[hub_type] = {
            'count': len(hub_data),
            'mean': hub_data.mean(),
            'median': hub_data.median(),
            'std': hub_data.std(),
            'min': hub_data.min(),
            'max': hub_data.max()
        }

    return summary_stats
def export_weight_median_plot(median_dict, filename='weight_medians.png', figsize=(12, 7)):
    """
    Creates and exports a bar plot of median weights for each scoring column.

    Args:
        median_dict: Dictionary containing median weights for each column
        filename: Name of the PNG file to export
        figsize: Tuple for figure size (width, height)
    """
    # Convert to pandas Series and sort
    median_series = pd.Series(median_dict).sort_values(ascending=False)

    # Create the bar plot
    plt.figure(figsize=figsize)
    bars = plt.bar(range(len(median_series)), median_series.values,
                   color='mediumseagreen', alpha=0.8, edgecolor='darkgreen', linewidth=1.2)

    # Customize the plot
    plt.xlabel('Scoring Columns', fontsize=12, fontweight='bold')
    plt.ylabel('Median Weight', fontsize=12, fontweight='bold')
    plt.title('Median Monte Carlo Weights by Scoring Column', fontsize=14, fontweight='bold', pad=20)

    # Set x-axis labels with better formatting
    column_labels = [col.replace('_', ' ').replace('Norm', '(Norm)') for col in median_series.index]
    plt.xticks(range(len(median_series)), column_labels, rotation=45, ha='right')

    # Add value labels on top of bars
    for i, (bar, value) in enumerate(zip(bars, median_series.values)):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{value:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

    # Add horizontal line at expected median (approximately 0.2 for 5 columns)
    expected_median = 1.0 / len(median_series)
    plt.axhline(y=expected_median, color='black', linestyle='--', alpha=0.7,
                label=f'Expected Value ({expected_median:.3f})')
    plt.legend()

    # Add grid for better readability
    plt.grid(axis='y', alpha=0.3, linestyle='--')

    # Set y-axis limits to show the full range nicely
    plt.ylim(0, max(median_series.values) * 1.15)

    # Adjust layout to prevent label cutoff
    plt.tight_layout()

    # Export as PNG
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Plot exported as {filename}")

    # Show the plot
    plt.show()

    return median_series
def export_weight_averages_plot(weight_dict, filename='weight_averages.png', figsize=(12, 7)):
    """
    Creates and exports a bar plot of average weights for each scoring column.

    Args:
        weight_dict: Dictionary containing average weights for each column
        filename: Name of the PNG file to export
        figsize: Tuple for figure size (width, height)
    """
    # Convert to pandas Series and sort
    weight_series = pd.Series(weight_dict).sort_values(ascending=False)

    # Create the bar plot
    plt.figure(figsize=figsize)
    bars = plt.bar(range(len(weight_series)), weight_series.values,
                   color='coral', alpha=0.8, edgecolor='darkred', linewidth=1.2)

    # Customize the plot
    plt.xlabel('Scoring Columns', fontsize=12, fontweight='bold')
    plt.ylabel('Average Weight', fontsize=12, fontweight='bold')
    plt.title('Average Monte Carlo Weights by Scoring Column', fontsize=14, fontweight='bold', pad=20)

    # Set x-axis labels with better formatting
    column_labels = [col.replace('_', ' ').replace('Norm', '(Norm)') for col in weight_series.index]
    plt.xticks(range(len(weight_series)), column_labels, rotation=45, ha='right')

    # Add value labels on top of bars
    for i, (bar, value) in enumerate(zip(bars, weight_series.values)):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{value:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

    # Add horizontal line at expected average (0.2 for 5 columns)
    expected_avg = 1.0 / len(weight_series)
    plt.axhline(y=expected_avg, color='black', linestyle='--', alpha=0.7,
                label=f'Expected Average ({expected_avg:.3f})')
    plt.legend()

    # Add grid for better readability
    plt.grid(axis='y', alpha=0.3, linestyle='--')

    # Set y-axis limits to show the full range nicely
    plt.ylim(0, max(weight_series.values) * 1.15)

    # Adjust layout to prevent label cutoff
    plt.tight_layout()

    # Export as PNG
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Plot exported as {filename}")

    # Show the plot
    plt.show()

    return weight_series

# Apply the Monte Carlo scoring function to the DataFrame
scored_mc, average_weights, median_weights, min_weights, max_weights = monte_carlo_scoring(scored.copy())

# Display the first few rows of the result
display(scored_mc.head())

# Export the bar plot showing average weights of each scoring column
weight_averages = export_weight_averages_plot(average_weights, 'weight_averages.png')

# Export the bar plot showing median weights of each scoring column
weight_medians = export_weight_median_plot(median_weights, 'weight_medians.png')

# Export the weight statistics line plot (average, min, max)
weight_stats_summary = export_weight_statistics_plot(average_weights, min_weights, max_weights, 'weight_statistics.png')

# Export the score distribution plot for each HubType
score_distribution_stats = export_score_distribution_plot(scored_mc, 'score_distribution.png')

# Print the statistics for reference
print("\nAverage Weights of Scoring Columns:")
print(weight_averages)
print("\nMedian Weights of Scoring Columns:")
print(weight_medians)
print("\nWeight Statistics Summary:")
print(weight_stats_summary)
print("\nScore Distribution Statistics by Hub Type:")
for hub_type, stats in score_distribution_stats.items():
    print(f"\n{hub_type}:")
    for stat_name, value in stats.items():
        if stat_name == 'count':
            print(f"  {stat_name}: {value}")
        else:
            print(f"  {stat_name}: {value:.4f}")

In [ ]:
scored_mc.to_csv('/content/drive/MyDrive/Hubs/Scored/Scored_Normalized_Weights_05112025.csv')

In [ ]:
display(scored_mc['Average_Simulated_Score'].describe())

In [ ]:
scored_mc['Rank_within_HubType'] = scored_mc.groupby('HubType')['Average_Simulated_Score'].rank(method='dense', ascending=False)

display(scored_mc.head())

In [ ]:
for hub_type in scored_mc['HubType'].unique():
  print(f"Top 5 Hubs for {hub_type}:")
  display(scored_mc[scored_mc['HubType'] == hub_type].sort_values('Rank_within_HubType').head())

In [ ]:
scored_mc.to_csv('/content/drive/MyDrive/Hubs/Scored/Scored_By_MonteCarlo_29-10-2025.csv', encoding='utf-8')

In [ ]:
scored_mc.head(1)

In [ ]:
scored_mc['h3_index'] = scored_mc['h3_index'].apply(lambda x: make_list(x))

In [ ]:
scored_mc[scored_mc.index==736]

In [ ]:
scored_mc = scored_mc.explode('h3_index')

In [ ]:
scored_mc

In [ ]:
scored_mc.to_csv('/content/drive/MyDrive/Hubs/Scored/h3_index_for_hub_naming_28102025.csv', encoding='utf-8')

# upload temp file to explode by h3_index

In [ ]:
old = pd.read_csv('/content/drive/MyDrive/Hubs/Hubs_Grouped_And_Scored.csv', encoding='utf-8')

In [ ]:
old.head(1)

In [ ]:
def make_list_old(txt):
  txt = txt.replace('{','').replace('}','').replace("'","").replace(' ','').split(',')
  return txt

In [ ]:
old['H3_Index'] = old['H3_Index'].apply(lambda x: make_list_old(x))

In [ ]:
old = old.explode('H3_Index')

In [ ]:
old.to_csv('/content/drive/MyDrive/Hubs/OLD_Hubs_Grouped_And_Scored_exploded.csv', encoding='utf-8')

In [ ]:
\[
\text{Given radii } \{r_i\}_{i=0}^{M} \text{ and } \beta>0,\quad
m_i \coloneqq \frac{r_{i-1}+r_i}{2}\quad (i=1,\dots,M),\quad
d_i \coloneqq m_i^{\beta}.
\]

\[
\text{Let } P_{k i} \text{ and } E_{k i} \text{ denote population and employment for row } k \text{ and ring } i,
\]
\[
C_{k i} \coloneqq w_{\text{pop}}\,P_{k i} + w_{\text{emp}}\,E_{k i}.
\]

\[
\boxed{\,s_k \;=\; \sum_{i=1}^{M} \frac{C_{k i}}{d_i}
\;=\; \sum_{i=1}^{M} \frac{w_{\text{pop}}\,P_{k i} + w_{\text{emp}}\,E_{k i}}{\left(\frac{r_{i-1}+r_i}{2}\right)^{\beta}}\,}
\qquad (k=1,\dots,N).
\]

\[
\textbf{(Matrix form)}\quad
\mathbf{s} \;=\; \bigl(w_{\text{pop}}\mathbf{P} + w_{\text{emp}}\mathbf{E}\bigr)\,\mathbf{D}^{-1}\,\mathbf{1}_M,
\]
where \(\mathbf{P},\mathbf{E}\in\mathbb{R}^{N\times M}\), \(\mathbf{D}=\mathrm{diag}(d_1,\dots,d_M)\), and \(\mathbf{1}_M\) is an \(M\)-vector of ones.


\[
\text{Given radii } \{r_i\}_{i=0}^{M} \text{ and } \beta>0,\quad
m_i \coloneqq \frac{r_{i-1}+r_i}{2}\quad (i=1,\dots,M),\quad
d_i \coloneqq m_i^{\beta}.
\]

\[
\text{Let } P_{k i} \text{ and } E_{k i} \text{ denote population and employment for row } k \text{ and ring } i,
\]
\[
C_{k i} \coloneqq w_{\text{pop}}\,P_{k i} + w_{\text{emp}}\,E_{k i}.
\]

\[
\boxed{\,s_k \;=\; \sum_{i=1}^{M} \frac{C_{k i}}{d_i}
\;=\; \sum_{i=1}^{M} \frac{w_{\text{pop}}\,P_{k i} + w_{\text{emp}}\,E_{k i}}{\left(\frac{r_{i-1}+r_i}{2}\right)^{\beta}}\,}
\qquad (k=1,\dots,N).
\]

\[
\textbf{(Matrix form)}\quad
\mathbf{s} \;=\; \bigl(w_{\text{pop}}\mathbf{P} + w_{\text{emp}}\mathbf{E}\bigr)\,\mathbf{D}^{-1}\,\mathbf{1}_M,
\]
where \(\mathbf{P},\mathbf{E}\in\mathbb{R}^{N\times M}\), \(\mathbf{D}=\mathrm{diag}(d_1,\dots,d_M)\), and \(\mathbf{1}_M\) is an \(M\)-vector of ones.


